In [ ]:
# Hapus proses streamlit dan ngrok yang mungkin masih berjalan di background
!pkill streamlit
!pkill ngrok

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

# --- CONFIGURATION ---
st.set_page_config(page_title="Shopping Trends Analytics", layout="wide", initial_sidebar_state="expanded")

# --- CUSTOM CSS (MODERN UI) ---
st.markdown("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
    html, body, [class*="css"] { font-family: 'Inter', sans-serif; }
    .main { background-color: #F8F9FB; }
    [data-testid="stSidebar"] { background-color: #0E1117; color: white; border-right: 1px solid #2D2E32; }
    [data-testid="stSidebar"] * { color: white !important; }
    .metric-card { background-color: white; padding: 20px; border-radius: 15px; box-shadow: 0 4px 6px rgba(0, 0, 0, 0.05); border: 1px solid #EDF0F5; margin-bottom: 20px; }
    .metric-title { color: #8E95A1; font-size: 14px; font-weight: 600; margin-bottom: 5px; }
    .metric-value { color: #1A1D23; font-size: 24px; font-weight: 700; }
    .stButton>button { border-radius: 10px; background-color: #FF2E63; color: white; border: none; width: 100%; height: 3em; font-weight: bold; }
    </style>
    """, unsafe_allow_html=True)

# --- LOAD DATA ---
@st.cache_data
def load_data():
    df = pd.read_csv('shopping_trends-selected-columns.csv')
    # Sederhanakan Map Koordinat
    state_coords = df.groupby('Location').size().reset_index()
    # Dummy lat/lon untuk visualisasi map jika data koordinat tidak ada
    np.random.seed(42)
    state_coords['lat'] = np.random.uniform(25, 48, len(state_coords))
    state_coords['lon'] = np.random.uniform(-120, -75, len(state_coords))
    df = df.merge(state_coords[['Location', 'lat', 'lon']], on='Location', how='left')
    return df

df = load_data()

# --- PREPROCESSING & MODEL TRAINING ---
# Kita tentukan fitur yang akan digunakan secara konsisten
FEATURES = ['Age', 'Gender', 'Item Purchased', 'Category', 'Location', 'Size', 'Color', 'Season']
TARGET = 'Purchase Amount (USD)'

@st.cache_resource
def train_all_models(data):
    df_ml = data.copy()
    encoders = {}

    # Label Encoding untuk kolom kategori
    cat_cols = ['Gender', 'Item Purchased', 'Category', 'Location', 'Size', 'Color', 'Season']
    for col in cat_cols:
        le = LabelEncoder()
        df_ml[col] = le.fit_transform(df_ml[col].astype(str))
        encoders[col] = le

    X = df_ml[FEATURES]
    y = df_ml[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Inisialisasi Model
    lr = LinearRegression().fit(X_train, y_train)
    rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)
    xgb = XGBRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)

    # Hitung Metrik
    results = []
    for name, m in [("Linear Regression", lr), ("Random Forest", rf), ("XGBoost", xgb)]:
        pred = m.predict(X_test)
        resulats.append({"Model": name, "MAE": mean_absolute_error(y_test, pred), "R2 Score": r2_score(y_test, pred)})

    return {"Linear Regression": lr, "Random Forest": rf, "XGBoost": xgb}, encoders, pd.DataFrame(results), X.columns

models_dict, encoders, metrics_df, feature_columns = train_all_models(df)

# --- SIDEBAR NAVIGATION ---
st.sidebar.markdown("## 🧭 Navigation")
page = st.sidebar.radio("Go to:", ["🏠 Home", "📦 Products", "📍 Locations", "🤖 Model Analysis"])

st.sidebar.markdown("---")
st.sidebar.markdown("## 🔍 Filters")
selected_cat = st.sidebar.multiselect("Category", df['Category'].unique(), default=df['Category'].unique())
filtered_df = df[df['Category'].isin(selected_cat)]

# --- PAGE: HOME ---
if page == "🏠 Home":
    st.title("Shopping Trends Overview 📊")
    m1, m2, m3 = st.columns(3)
    with m1:
        st.markdown(f"<div class='metric-card'><div class='metric-title'>TOTAL CUSTOMERS</div><div class='metric-value'>{len(filtered_df):,}</div></div>", unsafe_allow_html=True)
    with m2:
        st.markdown(f"<div class='metric-card'><div class='metric-title'>AVG. PURCHASE</div><div class='metric-value'>${filtered_df[TARGET].mean():.1f}</div></div>", unsafe_allow_html=True)
    with m3:
        st.markdown(f"<div class='metric-card'><div class='metric-title'>TOTAL SALES</div><div class='metric-value'>${filtered_df[TARGET].sum():,}</div></div>", unsafe_allow_html=True)

    c1, c2 = st.columns([2, 1])
    with c1:
        st.subheader("The Popularity of Each Category")
        fig_bar = px.histogram(filtered_df, x="Category", color="Category", color_discrete_sequence=px.colors.qualitative.Bold)
        st.plotly_chart(fig_bar, use_container_width=True)
    with c2:
        st.subheader("Gender Distribution")
        fig_pie = px.pie(filtered_df, names="Gender", hole=0.6, color_discrete_sequence=['#FF2E63', '#08D9D6'])
        st.plotly_chart(fig_pie, use_container_width=True)

# --- PAGE: PRODUCTS ---
elif page == "📦 Products":
    st.title("Product Analysis 🎒")
    c1, c2 = st.columns(2)
    with c1:
        st.subheader("Top 10 Products")
        top_items = filtered_df['Item Purchased'].value_counts().nlargest(10).reset_index()
        st.plotly_chart(px.bar(top_items, x='Item Purchased', y='count', color='Item Purchased'), use_container_width=True)
    with c2:
        st.subheader("Size Popularity")
        st.plotly_chart(px.pie(filtered_df, names="Size", hole=0.4), use_container_width=True)

# --- PAGE: LOCATIONS ---
elif page == "📍 Locations":
    st.title("Sales by Locations 🌎")
    st.subheader("Geographic Sales Distribution")
    fig_map = px.scatter_geo(filtered_df, lat='lat', lon='lon', color=TARGET, hover_name='Location', size=TARGET, projection="albers usa")
    st.plotly_chart(fig_map, use_container_width=True)

# --- PAGE: MODEL ANALYSIS ---
elif page == "🤖 Model Analysis":
    st.title("Model Performance & Prediction 🧠")

    # 1. Tampilkan Perbandingan Metrik
    st.subheader("Model Performance Comparison")
    st.table(metrics_df.set_index("Model"))

    st.markdown("---")

    # 2. Form Prediksi
    st.subheader("Predict Purchase Amount")
    st.info("Fill the details below to estimate the customer's spending.")

    with st.container():
        col1, col2, col3 = st.columns(3)
        with col1:
            in_age = st.slider("Age", 18, 75, 30)
            in_gen = st.selectbox("Gender", encoders['Gender'].classes_)
            in_cat = st.selectbox("Category", encoders['Category'].classes_)
        with col2:
            in_item = st.selectbox("Item Purchased", encoders['Item Purchased'].classes_)
            in_loc = st.selectbox("Location", encoders['Location'].classes_)
            in_size = st.selectbox("Size", encoders['Size'].classes_)
        with col3:
            in_col = st.selectbox("Color", encoders['Color'].classes_)
            in_seas = st.selectbox("Season", encoders['Season'].classes_)
            sel_model = st.selectbox("Select Model", list(models_dict.keys()))

    if st.button("Calculate Prediction"):
        # Susun input menjadi DataFrame
        input_data = pd.DataFrame([{
            'Age': in_age,
            'Gender': in_gen,
            'Item Purchased': in_item,
            'Category': in_cat,
            'Location': in_loc,
            'Size': in_size,
            'Color': in_col,
            'Season': in_seas
        }])

        # Transformasi input menggunakan encoder yang sudah disimpan
        for col in encoders:
            input_data[col] = encoders[col].transform(input_data[col].astype(str))

        # Pastikan urutan kolom sama dengan saat training
        input_data = input_data[feature_columns]

        # Prediksi
        prediction = models_dict[sel_model].predict(input_data)[0]

        st.markdown("### Result:")
        st.success(f"Estimated Purchase Amount: **${prediction:.2f}**")
        st.balloons()

Writing app.py


In [ ]:
# 1. Install dependencies
!pip install streamlit pyngrok xgboost -q

# 2. Setup Ngrok
from pyngrok import ngrok
import time

# Masukkan Token Anda
NGROK_AUTH_TOKEN = "3EEymuyn9AhG52PwQwC45RQPkhw_2eBwZYtFedbiRPZsC2u2E"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 3. Jalankan Streamlit (output akan muncul di bawah cell agar kita bisa cek error-nya)
import subprocess
process = subprocess.Popen(['streamlit', 'run', 'app.py'])

# Beri waktu streamlit untuk start
time.sleep(5)

# 4. Buka Tunnel
try:
    url = ngrok.connect(8501)
    print("Dashboard Berhasil Dijalankan!")
    print("KLIK LINK INI UNTUK MEMBUKA:")
    print(url.public_url)
except Exception as e:
    print("Terjadi kesalahan Ngrok:", e)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 77.7 MB/s eta 0:00:00
Dashboard Berhasil Dijalankan!
KLIK LINK INI UNTUK MEMBUKA:
https://rigor-deletion-diffusive.ngrok-free.dev
